In [1]:
import cv2
import numpy as np
import os
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_regression
from sklearn.svm import SVC,SVR
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.metrics import accuracy_score, mean_absolute_error
from sklearn.model_selection import train_test_split, cross_val_score
from tqdm import tqdm
from joblib import load,dump

In [ ]:
def extract_sift_features(images, max_features=128):
    sift = cv2.SIFT_create(nfeatures=max_features)
    features = []

    for img in tqdm(images, desc="Extracting SIFT"):
        keypoints, descriptors = sift.detectAndCompute(img, None)
        if descriptors is None:
            descriptors = np.zeros((1, 128))
        if descriptors.shape[0] > max_features:
            descriptors = descriptors[:max_features]
        elif descriptors.shape[0] < max_features:
            padding = np.zeros((max_features - descriptors.shape[0], 128))
            descriptors = np.vstack((descriptors, padding))
        features.append(descriptors.flatten())

    return np.array(features)

In [3]:
def select_k_best(X, y, k=100, task='classification'):
    if task == 'classification':
        selector = SelectKBest(score_func=f_classif, k=k)
    elif task == 'regression':
        selector = SelectKBest(score_func=mutual_info_regression, k=k)
    
    X_new = selector.fit_transform(X, y)
    return X_new, selector

In [4]:
def sift_and_select(X, y_age, y_gender, k=100, name='dataset'):
    X_sift = extract_sift_features(X)

    X_gender, selector_gender = select_k_best(X_sift, y_gender, k=k, task='classification')
    dump(selector_gender, f"models/model1_4/selector_gender_{name}.joblib")

    X_age, selector_age = select_k_best(X_sift, y_age, k=k, task='regression')
    dump(selector_age, f"models/model1_4/selector_age_{name}.joblib")

    print(f"{name}: Seçiciler kaydedildi.")
    return X_sift, X_gender, X_age


In [5]:
def train_and_save_models_sift(X_train, X_test, gen_train, gen_test, age_train, age_test, name='dataset'):

    # Özellik seçimi (SelectKBest ile)
    selector = SelectKBest(score_func=f_classif, k=200)
    X_train_selected = selector.fit_transform(X_train, gen_train)
    X_test_selected = selector.transform(X_test)

    dump(selector, f"models/model1_4/selector_{name}.joblib")
    print(f"{name}: SelectKBest özelliği kaydedildi.")

    print(f"\n=== {name.upper()} - SVM / SVR ===")
    svm = SVC(kernel='rbf', C=1.0, random_state=42)
    svr = SVR(kernel='rbf', C=1.0)

    cv_acc = cross_val_score(svm, X_train_selected, gen_train, cv=5)
    print("SVM Gender CV Accuracy:", np.mean(cv_acc))
    cv_mae = cross_val_score(svr, X_train_selected, age_train, cv=5, scoring='neg_mean_absolute_error')
    print("SVR Age CV MAE:", -np.mean(cv_mae))

    svm.fit(X_train_selected, gen_train)
    svr.fit(X_train_selected, age_train)
    test_acc = svm.score(X_test_selected, gen_test)
    test_mae = np.mean(np.abs(svr.predict(X_test_selected) - age_test))
    print("SVM Gender Test Accuracy:", test_acc)
    print("SVR Age Test MAE:", test_mae)

    dump(svm, f"models/model1_4/svm_gender_{name}.joblib")
    dump(svr, f"models/model1_4/svr_age_{name}.joblib")
    print(f"{name}: SVM & SVR modelleri kaydedildi.")

    print(f"\n=== {name.upper()} - MLP ===")
    mlp_clf = MLPClassifier(hidden_layer_sizes=(50,), activation='relu', max_iter=2000, random_state=42)
    mlp_reg = MLPRegressor(hidden_layer_sizes=(50,), activation='relu', max_iter=2000, random_state=42)

    cv_acc = cross_val_score(mlp_clf, X_train_selected, gen_train, cv=5)
    print("MLP Gender CV Accuracy:", np.mean(cv_acc))
    cv_mae = cross_val_score(mlp_reg, X_train_selected, age_train, cv=5, scoring='neg_mean_absolute_error')
    print("MLP Age CV MAE:", -np.mean(cv_mae))

    mlp_clf.fit(X_train_selected, gen_train)
    mlp_reg.fit(X_train_selected, age_train)
    test_acc = mlp_clf.score(X_test_selected, gen_test)
    test_mae = np.mean(np.abs(mlp_reg.predict(X_test_selected) - age_test))
    print("MLP Gender Test Accuracy:", test_acc)
    print("MLP Age Test MAE:", test_mae)

    dump(mlp_clf, f"models/model1_4/mlp_gender_{name}.joblib")
    dump(mlp_reg, f"models/model1_4/mlp_age_{name}.joblib")
    print(f"{name}: MLP modelleri kaydedildi.")

    print(f"\n=== {name.upper()} - RANDOM FOREST ===")
    rf_clf = RandomForestClassifier(n_estimators=75, random_state=42)
    rf_reg = RandomForestRegressor(n_estimators=75, random_state=42)

    cv_acc = cross_val_score(rf_clf, X_train_selected, gen_train, cv=5)
    print("RF Gender CV Accuracy:", np.mean(cv_acc))
    cv_mae = cross_val_score(rf_reg, X_train_selected, age_train, cv=5, scoring='neg_mean_absolute_error')
    print("RF Age CV MAE:", -np.mean(cv_mae))

    rf_clf.fit(X_train_selected, gen_train)
    rf_reg.fit(X_train_selected, age_train)
    test_acc = rf_clf.score(X_test_selected, gen_test)
    test_mae = np.mean(np.abs(rf_reg.predict(X_test_selected) - age_test))
    print("RF Gender Test Accuracy:", test_acc)
    print("RF Age Test MAE:", test_mae)

    dump(rf_clf, f"models/model1_4/rf_gender_{name}.joblib")
    dump(rf_reg, f"models/model1_4/rf_age_{name}.joblib")
    print(f"{name}: Random Forest modelleri kaydedildi.")


In [6]:
def test_model_sift(X_test, y_age_test, y_gender_test, name='dataset'):
    model_dir = f'models/model1_4/'

    # Model ve SelectKBest (özellik seçici) yükleniyor
    selector = load(f'{model_dir}selector_{name}.joblib')

    # SIFT özellikleri çıkarılıyor (önceden yazılmış fonksiyon)
    X_sift = extract_sift_features(X_test)  # Bu fonksiyonu sen tanımlamış olmalısın
    X_selected = selector.transform(X_sift)

    print(f"\n=== [ {name.upper()} ] TEST SONUÇLARI ===")

    # --- SVM + SVR ---
    svm = load(f'{model_dir}svm_gender_{name}.joblib')
    svr = load(f'{model_dir}svr_age_{name}.joblib')
    acc_svm = svm.score(X_selected, y_gender_test)
    mae_svr = mean_absolute_error(y_age_test, svr.predict(X_selected))
    print(f"[SVM] Gender Accuracy: {acc_svm:.3f}")
    print(f"[SVR] Age MAE: {mae_svr:.2f}")

    # --- MLP ---
    mlp_clf = load(f'{model_dir}mlp_gender_{name}.joblib')
    mlp_reg = load(f'{model_dir}mlp_age_{name}.joblib')
    acc_mlp = mlp_clf.score(X_selected, y_gender_test)
    mae_mlp = mean_absolute_error(y_age_test, mlp_reg.predict(X_selected))
    print(f"[MLP] Gender Accuracy: {acc_mlp:.3f}")
    print(f"[MLP] Age MAE: {mae_mlp:.2f}")

    # --- Random Forest ---
    rf_clf = load(f'{model_dir}rf_gender_{name}.joblib')
    rf_reg = load(f'{model_dir}rf_age_{name}.joblib')
    acc_rf = rf_clf.score(X_selected, y_gender_test)
    mae_rf = mean_absolute_error(y_age_test, rf_reg.predict(X_selected))
    print(f"[RF]  Gender Accuracy: {acc_rf:.3f}")
    print(f"[RF]  Age MAE: {mae_rf:.2f}")

In [7]:
# Eğitim verisi
X_train = np.load("X_train_utkface.npy", allow_pickle=True)
y_age_train = np.load("y_age_train_utkface.npy")
y_gen_train = np.load("y_gender_train_utkface.npy")

# Test verisi
X_test = np.load("X_test_utkface.npy", allow_pickle=True)
y_age_test = np.load("y_age_test_utkface.npy")
y_gen_test = np.load("y_gender_test_utkface.npy")

In [ ]:
X_train_sift, X_train_gender, X_train_age = sift_and_select(X_train, y_age_train, y_gen_train, name="UTKface_1")

# Test için SIFT çıkarımı
X_test_sift = extract_sift_features(X_test)

# Test için iki ayrı seçici yükle
selector_gender = load("models/model1_4/selector_gender_UTKface_1.joblib")
selector_age = load("models/model1_4/selector_age_UTKface_1.joblib")

X_test_gender = selector_gender.transform(X_test_sift)
X_test_age = selector_age.transform(X_test_sift)

Extracting SIFT: 100%|██████████| 19044/19044 [01:04<00:00, 296.69it/s]


In [ ]:
train_and_save_models_sift(
    X_train_gender, X_test_gender,
    y_gen_train, y_gen_test,
    y_age_train, y_age_test,
    name="UTKface_1"
)

In [ ]:
test_model_sift(X_test, y_age_test, y_gen_test, name="UTKface_1")
